# Défi 2 NLP (medRxiv 2026) — Projet 2 : AI-Written Abstract Detection
**Approche :** Deep Learning (Fine-tuning de `BiomedBERT` avec perte pondérée & calibrage du seuil F1)  
**Auteur :** Redouane Kenniche

---

### Objectif du notebook
Ce notebook adapte le modèle neuronal pré-entraîné sur la littérature biomédicale (`microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract`) à la détection d'abstracts générés par IA (`0` = Humain, `1` = IA). Il intègre une pondération inverse des fréquences de classes et une optimisation du seuil de décision pour maximiser le **F1-score**.

## 1. Importation des bibliothèques et configuration GPU
Chargement de PyTorch, Hugging Face Transformers et Scikit-learn, et détection automatique du GPU (`cuda`).

In [1]:
import html
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score, precision_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
print(f"Périphérique utilisé : {DEVICE}")

Périphérique utilisé : cuda


## 2. Chargement et nettoyage léger des textes
Localisation automatique des fichiers d'entraînement et de test, puis nettoyage minimal conservant la ponctuation et les chiffres essentiels à la détection de style IA.

In [2]:
def trouver_fichier(noms_possibles):
    cibles = set(noms_possibles)
    for racine, _, fichiers in os.walk("."):
        for f in fichiers:
            if f in cibles:
                return os.path.join(racine, f)
    return None


def lire_csv_robuste(filepath):
    for enc in ["utf-8", "latin-1", "cp1252"]:
        for sep in [",", ";"]:
            try:
                df = pd.read_csv(filepath, encoding=enc, sep=sep)
                df.columns = df.columns.str.strip().str.strip('"')
                if "abstract" in {c.lower() for c in df.columns}:
                    return df
            except Exception:
                continue
    raise ValueError(f"Impossible de lire : {filepath}")


def nettoyer_html_leger(texte):
    if not isinstance(texte, str):
        return ""
    texte = html.unescape(texte)
    texte = re.sub(
        r"</?(?:p|b|i|u|sub|sup|br|em|strong|span|div|ul|ol|li)\b[^>]*>",
        " ",
        texte,
    )
    return re.sub(r"\s+", " ", texte).strip()


TRAIN_FILE = trouver_fichier([
    "ClassificationsHumanAI_AbstractsTrainSet.csv",
    "ClassificationsHumanAI_AbstractsTrainSet.csv.gz",
])
TEST_FILE = trouver_fichier([
    "ClassificationsHumanAI_AbstractsTestSet.csv",
    "ClassificationsHumanAI_AbstractsTestSet.csv.gz",
])

df_train = lire_csv_robuste(TRAIN_FILE)
col_map = {c.lower(): c for c in df_train.columns}
col_abstract = col_map["abstract"]
col_target = col_map.get("label", col_map.get("humanai"))

df_train = df_train.dropna(subset=[col_abstract, col_target]).reset_index(drop=True)
df_train["clean_abstract"] = df_train[col_abstract].apply(nettoyer_html_leger)
df_train["label"] = df_train[col_target].astype(int)
df_train = df_train.drop_duplicates(subset=["clean_abstract"]).reset_index(drop=True)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_train["clean_abstract"].tolist(),
    df_train["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_train["label"].tolist(),
)
print(f"Train : {len(train_texts)} | Validation : {len(val_texts)}")

Train : 6710 | Validation : 1678


## 3. Tokenisation et création des Datasets PyTorch
Encodage des résumés avec le tokenizer de `BiomedBERT` (longueur maximale de 384 tokens) et calcul des poids de classes pour compenser le déséquilibre entre textes humains (`0`) et textes IA (`1`).

In [3]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_labels,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class HumanAIDataset(Dataset):

    def __init__(self, texts, labels=None, max_len=384):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True, max_length=max_len
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])


train_ds = HumanAIDataset(train_texts, train_labels)
val_ds = HumanAIDataset(val_texts, val_labels)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

## 4. Entraînement de BiomedBERT et calibrage du seuil F1
Fine-tuning sur 3 époques avec sauvegarde automatique du meilleur checkpoint selon le **F1-score**, suivi de la recherche du seuil de probabilité optimal sur le jeu de validation.

In [4]:
class WeightedTrainer(Trainer):

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
    }


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

training_args = TrainingArguments(
    output_dir="./results_projet2",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

# Recherche du seuil optimal sur les probabilités de validation
val_logits = trainer.predict(val_ds).predictions
val_probs = torch.softmax(torch.tensor(val_logits), dim=-1)[:, 1].numpy()

best_thresh, best_f1, best_prec = 0.5, -1.0, 0.0
for thresh in np.arange(0.25, 0.80, 0.01):
    preds = (val_probs >= thresh).astype(int)
    f1 = f1_score(val_labels, preds)
    prec = precision_score(val_labels, preds, zero_division=0)
    if (f1 > best_f1) or (np.isclose(f1, best_f1) and prec > best_prec):
        best_thresh, best_f1, best_prec = float(thresh), float(f1), float(prec)

val_preds = (val_probs >= best_thresh).astype(int)
print(
    f"Validation BiomedBERT | Seuil optimal : {best_thresh:.2f} | "
    f"F1-score : {best_f1:.4f} | Précision : {best_prec:.4f}"
)
print(classification_report(val_labels, val_preds, target_names=["Humain (0)", "IA (1)"]))

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical ar

Epoch,Training Loss,Validation Loss,F1,Precision
1,0.171477,0.162917,0.869379,0.791423
2,0.161065,0.166201,0.889858,0.822581
3,0.036088,0.202202,0.916188,0.886667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Validation BiomedBERT | Seuil optimal : 0.72 | F1-score : 0.9167 | Précision : 0.8939
              precision    recall  f1-score   support

  Humain (0)       0.98      0.96      0.97      1257
      IA (1)       0.89      0.94      0.92       421

    accuracy                           0.96      1678
   macro avg       0.94      0.95      0.94      1678
weighted avg       0.96      0.96      0.96      1678



## 5. Inférence sur le jeu de test et export des prédictions
Application du modèle fine-tuné et du seuil optimal sur `ClassificationsHumanAI_AbstractsTestSet.csv`, puis sauvegarde du fichier `Projet2_Script2_DL.csv` (`ID`, `pred`).

In [5]:
if TEST_FILE:
    df_test = lire_csv_robuste(TEST_FILE)
    test_col_map = {c.lower(): c for c in df_test.columns}
    col_id_test = test_col_map.get("id", df_test.columns[0])
    col_abs_test = test_col_map["abstract"]

    test_ds = HumanAIDataset(
        df_test[col_abs_test].fillna("").apply(nettoyer_html_leger).tolist()
    )
    test_logits = trainer.predict(test_ds).predictions
    test_probs = torch.softmax(torch.tensor(test_logits), dim=-1)[:, 1].numpy()
    test_preds = (test_probs >= best_thresh).astype(int)

    df_sub = pd.DataFrame({
        "ID": df_test[col_id_test],
        "pred": test_preds,
    })
    df_sub.to_csv("Projet2_Script2_DL.csv", index=False)
    print(f"Fichier généré : Projet2_Script2_DL.csv ({df_sub.shape})")
    print("Distribution prédite :", df_sub["pred"].value_counts().to_dict())
else:
    print("Fichier de test introuvable dans :", os.listdir("."))

Fichier généré : Projet2_Script2_DL.csv ((2898, 2))
Distribution prédite : {0: 2213, 1: 685}
